# 01. Quickstart

Load the recommended HOMER π, query a mouse region, see its top-K human partners with their multi-source trust tier.

This notebook does **not** re-fit the model, it loads pre-computed outputs from `outputs/coupling/`. To regenerate, run `experiments/anchor_packs/compose_all.py` then this notebook again.

**Sections:**
1. Setup, load π, atlases, trust map
2. Single-region query (interactive)
3. Compare two π files (production vs production+packs)
4. Bulk region translation
5. 3D brain visualisation

In [1]:
# Setup
import sys, warnings
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / 'src'))
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from homer.data import load_cached

ROOT = Path.cwd().parent
ANN = ROOT / 'outputs' / 'anndata'
COUP = ROOT / 'outputs' / 'coupling'

# Load atlases (instant, h5ad caches)
M, _ = load_cached('mouse', cache_dir=ANN)
H, _ = load_cached('human', cache_dir=ANN)
print(f'Mouse parcels: {len(M.var)}, Human parcels: {len(H.var)}')

# Load the two production π
pi_strict = np.load(COUP / 'pi_fc_plus_SC.npy')
pi_packed = np.load(COUP / 'pi_fc_plus_SC_with_all_packs.npy')
print(f'π strict (Garin anchors only):     {pi_strict.shape}')
print(f'π packed (+ 5 default anchor packs): {pi_packed.shape}')

# Load multi-source trust map for the packed π
trust = np.load(COUP / 'trust_multisource_all_packs.npz', allow_pickle=True)
trust_tier = trust['evidence_tier']
print(f'\nTrust tiers (over {len(trust_tier)} mouse parcels):')
for t in ['anchored_and_validated', 'anchored_only', 'validated_only', 'structural', 'low_evidence']:
    n = int((trust_tier == t).sum())
    print(f'  {t:25s} {n:4d}  ({n/len(trust_tier):>6.1%})')

Mouse parcels: 1864, Human parcels: 2094
π strict (Garin anchors only):     (1864, 2094)
π packed (+ 5 default anchor packs): (1864, 2094)

Trust tiers (over 1864 mouse parcels):
  anchored_and_validated     571  ( 30.6%)
  anchored_only              240  ( 12.9%)
  validated_only             448  ( 24.0%)
  structural                 241  ( 12.9%)
  low_evidence               364  ( 19.5%)


## 2. Query a single mouse region, interactive

Pick a mouse region from the dropdown. Adjust top-K with the slider. The display shows:
- the top-K human partners with their MNI coords and region names
- the predicted region's trust tier (multi-source evidence label)
- the row's mass concentration (sharpness of the prediction)

In [2]:
import ipywidgets as widgets
from IPython.display import display, clear_output

# Build a sorted list of unique mouse regions (most-represented first)
region_counts = M.var['region'].value_counts()
mouse_regions = region_counts.index.tolist()

# Widgets
region_dropdown = widgets.Dropdown(
    options=mouse_regions, value=mouse_regions[0],
    description='Mouse region:', layout=widgets.Layout(width='500px'),
    style={'description_width': '120px'},
)
parcel_dropdown = widgets.Dropdown(
    options=[], description='Mouse parcel:',
    layout=widgets.Layout(width='500px'),
    style={'description_width': '120px'},
)
topk_slider = widgets.IntSlider(value=5, min=1, max=10, step=1,
                                  description='top-K:',
                                  style={'description_width': '120px'})
pi_choice = widgets.Dropdown(options=[('production + all packs', 'packed'),
                                        ('production point-anchor only', 'strict')],
                              value='packed', description='Which π:',
                              style={'description_width': '120px'},
                              layout=widgets.Layout(width='500px'))
out = widgets.Output()

def update_parcels(*_):
    region = region_dropdown.value
    parcels = M.var[M.var['region'] == region].index.tolist()
    parcel_dropdown.options = parcels
    parcel_dropdown.value = parcels[0] if parcels else None

def render(*_):
    with out:
        clear_output()
        parcel_id = parcel_dropdown.value
        if parcel_id is None:
            print('No parcel selected.')
            return
        pi = pi_packed if pi_choice.value == 'packed' else pi_strict
        # Find the row index for this parcel
        row = M.var.index.get_loc(parcel_id)
        k = topk_slider.value
        topk = pi[row].argsort()[::-1][:k]
        scores = pi[row][topk]

        tier = trust_tier[row] if pi_choice.value == 'packed' else 'n/a (strict π)'
        concentration = pi[row].max() / pi[row].sum()

        print(f'Mouse parcel: {parcel_id} ({M.var.iloc[row]["region"]})')
        print(f'  xyz: ({M.var.iloc[row].x:+.2f}, {M.var.iloc[row].y:+.2f}, {M.var.iloc[row].z:+.2f}) mm')
        print(f'  Trust tier:        {tier}')
        print(f'  Row concentration: {concentration:.1%} (peak / row sum)')
        print(f'\nTop {k} human partners ({pi_choice.value} π):')
        rows = []
        for r in topk:
            rows.append({
                'human_parcel': H.var.index[r],
                'region': H.var.iloc[r]['region'],
                'x':  f'{H.var.iloc[r].x:+.1f}',
                'y':  f'{H.var.iloc[r].y:+.1f}',
                'z':  f'{H.var.iloc[r].z:+.1f}',
                'pi':       f'{pi[row, r]:.4f}',
                'pi_frac':  f'{pi[row, r] / pi[row].sum():.1%}',
            })
        display(pd.DataFrame(rows))

region_dropdown.observe(update_parcels, names='value')
update_parcels()
for w in (region_dropdown, parcel_dropdown, topk_slider, pi_choice):
    w.observe(render, names='value')

display(widgets.VBox([region_dropdown, parcel_dropdown, topk_slider, pi_choice, out]))
render()

## 3. Compare predictions between strict π and packed π

For the selected parcel above, run the cell below to see how strict (Garin-only) and packed (with 5 anchor packs) predictions differ side-by-side. This makes the effect of anchor packs visible per-parcel.

In [3]:
def side_by_side(parcel_id, k=5):
    row = M.var.index.get_loc(parcel_id)
    top_strict = pi_strict[row].argsort()[::-1][:k]
    top_packed = pi_packed[row].argsort()[::-1][:k]
    table = []
    for i in range(k):
        rs, rp = top_strict[i], top_packed[i]
        table.append({
            'rank': i + 1,
            'strict region':    H.var.iloc[rs]['region'][:30],
            'strict π':          f'{pi_strict[row, rs]:.4f}',
            'packed region':    H.var.iloc[rp]['region'][:30],
            'packed π':          f'{pi_packed[row, rp]:.4f}',
        })
    return pd.DataFrame(table)

# Example: see how Motor parcel mapping differs
motor_parcel = M.var[M.var['region'].str.contains('Motor', case=False, na=False)].index[0]
print(f'Mouse parcel {motor_parcel} ({M.var.loc[motor_parcel, "region"]}):')
side_by_side(motor_parcel, k=5)

Mouse parcel 3 (L_Motor and premotor):


,rank,strict region,strict π,packed region,packed π
0,1,L_Motor and premotor,0.0005,L_1002,0.0002
1,2,R_1047,0.0000,L_1040,0.0002
2,3,R_348,0.0000,L_1004,0.0000
3,4,R_351,0.0000,L_946,0.0000
4,5,L_351,0.0000,R_1002,0.0000


## 4. Bulk region translation

For a whole mouse region, aggregate π across all its member parcels and show the top human regions.

In [4]:
def translate_region(region_query, pi=pi_packed, top_k=5):
    """Aggregate π over mouse parcels matching region_query, return top-K human regions."""
    mask = M.var['region'].str.contains(region_query, case=False, na=False)
    if mask.sum() == 0:
        return f'No mouse parcels match {region_query!r}'
    pi_M = pi[mask].sum(axis=0)
    pi_M /= pi_M.sum()
    # Aggregate by human region name
    h_regions = H.var['region'].copy()
    agg = pd.Series(pi_M).groupby(h_regions.values).sum().sort_values(ascending=False).head(top_k)
    out = pd.DataFrame({
        'human_region': agg.index,
        'mass_share':   [f'{v:.1%}' for v in agg.values],
    })
    print(f'Mouse "{region_query}", {int(mask.sum())} parcels, top-{top_k} human partners:')
    return out

translate_region('Hippocampal', top_k=10)

"No mouse parcels match 'Hippocampal'"

Try other regions:

In [5]:
# Visualise where Motor maps to
print(translate_region('Motor', top_k=8))
print('---')
print(translate_region('Amygdala', top_k=8))
print('---')
print(translate_region('Thalamus', top_k=8))

Mouse "Motor" — 2 parcels — top-8 human partners:
  human_region mass_share
0       R_1002      29.1%
1       L_1002      23.9%
2       L_1040      18.0%
3       R_1040      11.7%
4       R_1004       3.4%
5       L_1004       2.8%
6        L_946       2.4%
7        R_997       2.3%
---
Mouse "Amygdala" — 2 parcels — top-8 human partners:
         human_region mass_share
0          L_Amygdala      50.0%
1          R_Amygdala      50.0%
2               L_145       0.0%
3               R_145       0.0%
4               R_151       0.0%
5               L_151       0.0%
6  R_Olfactory cortex       0.0%
7  L_Olfactory cortex       0.0%
---
Mouse "Thalamus" — 4 parcels — top-8 human partners:
     human_region mass_share
0  L_Hypothalamus      25.0%
1      L_Thalamus      25.0%
2      R_Thalamus      25.0%
3  R_Hypothalamus      25.0%
4           R_346       0.0%
5           L_346       0.0%
6           R_340       0.0%
7           L_340       0.0%


## 5. 3D brain view

Render the mouse brain coloured by Garin functional network. Each dot is a parcel.

In [6]:
from homer.viz.notebook import plot_brain_3d
plot_brain_3d(M, color_by='network', title='Mouse atlas, network coloring')

## What's next

- See `fig1_coupling.ipynb` for the per-parcel evidence tiers and what they grade.
- See `fig2_what_carries_homology.ipynb` for what the anchors actually buy (parcel precision, not region-level correspondence).
- See `02_methodology.ipynb` for the FGW solver step-by-step.
- See `docs/04_anchor_packs.md` for how to add new packs.

For programmatic use, see the snippet in `README.md`.

## Manuscript figures — Fig 1b (coupling) & Fig 1c (worked query)

In [ ]:
# >>> MANUSCRIPT FIGURE  (regenerates the paper panel for this dataset)
import os, sys, json
from pathlib import Path
import numpy as np, matplotlib.pyplot as plt
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
FIG = ROOT.parent / 'manuscript' / 'figures'; FIG.mkdir(parents=True, exist_ok=True)
def L(f): return json.load(open(ROOT / 'outputs' / 'logs' / f))
plt.rcParams.update({'font.size':10,'axes.spines.top':False,'axes.spines.right':False,'savefig.dpi':300,'savefig.bbox':'tight'})
BLUE,GREY,ORANGE,PURPLE='#1b4f8a','#c9c9c9','#e08a2b','#6a51a3'
sys.path.insert(0, str(ROOT/'src'))
from homer.data import load_cached, load_pi
pi = load_pi(); M,_ = load_cached('mouse', cache_dir=ROOT/'outputs/anndata'); H,_ = load_cached('human', cache_dir=ROOT/'outputs/anndata')
hx,hy,hz = H.var['x'].to_numpy(), H.var['y'].to_numpy(), H.var['z'].to_numpy()

mo=np.argsort(M.var['pairid'].to_numpy()); ho=np.argsort(H.var['pairid'].to_numpy())
fig,ax=plt.subplots(figsize=(5.6,5.0)); ax.imshow(np.log10(pi[np.ix_(mo,ho)]+1e-6),aspect='auto',cmap='magma',vmin=-6,vmax=-2)
ax.set_xlabel('human parcels (by region)'); ax.set_ylabel('mouse parcels (by region)')
ax.set_title('Coupling π (1,864 × 2,094)',fontweight='bold',loc='left'); fig.savefig(FIG/'fig1b_coupling.png'); plt.show()

reg=M.var['region'].astype(str).str.lower(); mset=np.where(reg.str.contains('motor'))[0]
dist=pi[mset].sum(0); dist/=dist.sum(); order=np.argsort(dist)[::-1]
fig,axes=plt.subplots(1,2,figsize=(9.2,4.2),gridspec_kw={'width_ratios':[1.3,1]})
sc=axes[0].scatter(hy,hz,s=8+900*dist/dist.max(),c=dist,cmap='viridis',alpha=0.8,edgecolor='none')
axes[0].set_xlabel('y (P→A)'); axes[0].set_ylabel('z (I→S)'); axes[0].set_title('Mouse primary motor → human coupling mass',fontweight='bold',loc='left')
fig.colorbar(sc,ax=axes[0],fraction=0.046,label='π mass')
seen={};
for j in order:
    nm=str(H.var['region'].iloc[j]); seen[nm]=seen.get(nm,0)+dist[j]
top=sorted(seen.items(),key=lambda kv:-kv[1])[:10][::-1]
axes[1].barh(range(len(top)),[v for _,v in top],color=BLUE); axes[1].set_yticks(range(len(top))); axes[1].set_yticklabels([k[:26] for k,_ in top],fontsize=8)
axes[1].set_xlabel('summed π mass'); axes[1].set_title('Top human regions',fontweight='bold',loc='left'); fig.tight_layout(); fig.savefig(FIG/'fig1c_query.png'); plt.show()


In [ ]:
# >>> MANUSCRIPT FIGURE  (extra session panels)
# Fig 1 additions: topographic preservation (coupling respects spatial layout) + mapping sharpness/posteriors (calibrated distribution). Also written to fig1/.
import subprocess, sys, os
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
env = dict(os.environ, PYTHONPATH='src' + os.pathsep + os.environ.get('PYTHONPATH',''))
_scripts = [
    '../manuscript/figures/fig_extra/make_topographic_preservation.py',
    '../manuscript/figures/fig_extra/make_uncertainty.py',
    '../manuscript/figures/fig1/make_fig1_motivation.py',
]
for _s in _scripts:
    print('running', _s)
    subprocess.run([sys.executable, _s], cwd=ROOT, env=env, check=True)
print('done — panels written under ../manuscript/figures/')
